In [0]:
print("Hello Databricks!")
print("Welcome to Retail Sales Data Engineering Project")

Hello Databricks!
Welcome to Retail Sales Data Engineering Project


In [0]:
data = [
    (1, "Laptop", 55000),
    (2, "Mobile", 25000),
    (3, "Keyboard", 1500),
    (4, "Mouse", 800)
]

columns = ["Product_ID", "Product_Name", "Price"]

df = spark.createDataFrame(data, columns)

display(df)

Product_ID,Product_Name,Price
1,Laptop,55000
2,Mobile,25000
3,Keyboard,1500
4,Mouse,800


In [0]:
file_path = "/Volumes/workspace/default/retail_sales_volume/retail_sales.csv"

print(file_path)

/Volumes/workspace/default/retail_sales_volume/retail_sales.csv


In [0]:
file_path = "/Volumes/workspace/default/retail_sales_volume/retail_sales.csv"

print(file_path)

/Volumes/workspace/default/retail_sales_volume/retail_sales.csv


In [0]:
file_path = "/Volumes/workspace/default/retail_sales_volume/retail_sales.csv.txt"

df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(df)

Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Price,City
1001,2026-01-05,C101,Laptop,Electronics,1,55000,Pune
1002,2026-01-06,C102,Mobile,Electronics,2,25000,Mumbai
1003,2026-01-07,C103,Keyboard,Accessories,3,1500,Pune
1004,2026-01-08,C101,Mouse,Accessories,2,800,Delhi
1005,2026-01-09,C104,Monitor,Electronics,1,12000,Mumbai
1006,2026-01-10,C105,Laptop,Electronics,1,55000,Bangalore
1006,2026-01-10,C105,Laptop,Electronics,1,55000,Bangalore
1007,2026-01-11,C106,Headphones,Accessories,null,2000,Pune
1008,2026-01-12,null,Tablet,Electronics,1,18000,Delhi
1009,2026-01-13,C108,Mouse,Accessories,2,800,null


In [0]:
# Check the number of records
print("Total records:", df.count())

# Check column names
print("Columns:", df.columns)

# Check data types
df.printSchema()

# Display first 5 records
display(df.limit(5))

Total records: 11
Columns: ['Order_ID', 'Order_Date', 'Customer_ID', 'Product', 'Category', 'Quantity', 'Price', 'City']
root
 |-- Order_ID: integer (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price: integer (nullable = true)
 |-- City: string (nullable = true)



Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Price,City
1001,2026-01-05,C101,Laptop,Electronics,1,55000,Pune
1002,2026-01-06,C102,Mobile,Electronics,2,25000,Mumbai
1003,2026-01-07,C103,Keyboard,Accessories,3,1500,Pune
1004,2026-01-08,C101,Mouse,Accessories,2,800,Delhi
1005,2026-01-09,C104,Monitor,Electronics,1,12000,Mumbai


In [0]:

from delta.tables import DeltaTable

file_path = "/Volumes/workspace/default/retail_sales_volume/retail_sales.csv.txt"

# Read incoming CSV
df = (spark.read.format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load(file_path))

table_name = "workspace.default.bronze_retail_sales"

# Create Bronze table on first run
if not spark.catalog.tableExists(table_name):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

else:
    # Insert only new Order_ID records
    delta_table = DeltaTable.forName(spark, table_name)

    (delta_table.alias("target")
     .merge(
         df.alias("source"),
         "target.Order_ID = source.Order_ID"
     )
     .whenNotMatchedInsertAll()
     .execute())

print("Incremental Bronze loading completed.")


Incremental Bronze loading completed.


In [0]:
%sql
SELECT *
FROM workspace.default.bronze_retail_sales;

Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Price,City
1001,2026-01-05,C101,Laptop,Electronics,1,55000,Pune
1002,2026-01-06,C102,Mobile,Electronics,2,25000,Mumbai
1003,2026-01-07,C103,Keyboard,Accessories,3,1500,Pune
1004,2026-01-08,C101,Mouse,Accessories,2,800,Delhi
1005,2026-01-09,C104,Monitor,Electronics,1,12000,Mumbai
1006,2026-01-10,C105,Laptop,Electronics,1,55000,Bangalore
1006,2026-01-10,C105,Laptop,Electronics,1,55000,Bangalore
1007,2026-01-11,C106,Headphones,Accessories,null,2000,Pune
1008,2026-01-12,null,Tablet,Electronics,1,18000,Delhi
1009,2026-01-13,C108,Mouse,Accessories,2,800,null
